# Production Inference Pipeline
Loads the `.joblib` model weights generated by `training.ipynb` and runs the full hybrid forecast on incoming data.

**Pipeline:** KNN Imputatio → IQR + Isolation Forest Anomaly Imputation → Prophet Baseline → LightGBM Residual Correction

## 1. Setup & Model Loading

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import joblib
import shap

warnings.filterwarnings('ignore')

MODELS_DIR = '../Models'
OUTPUT_DIR = '../Outputs'

TARGET_COL = 'Demand_MWh'

# Must match the features used during training in hybrid_model.py
FEATURE_CANDIDATES = [
    'Day_of_Week', 'Is_Weekend', 'Is_Holiday',
    'Month', 'DayOfYear', 'WeekOfYear', 'Trend',
    'Avg_Temp', 'Rainfall', 'Temp_Lag_1',
    'Lag_1', 'Lag_2', 'Lag_7', 'Lag_14', 'Lag_30',
    'Rolling_7', 'Rolling_14', 'Rolling_30',
]

In [ ]:
# Load serialized models
try:
    prophet_model = joblib.load(os.path.join(MODELS_DIR, 'prophet_model.joblib'))
    lgbm_model = joblib.load(os.path.join(MODELS_DIR, 'lgbm_model.joblib'))
    iso_forest = joblib.load(os.path.join(MODELS_DIR, 'iso_forest.joblib'))
    knn_imputer = joblib.load(os.path.join(MODELS_DIR, 'knn_imputer.joblib'))
    print('SUCCESS: Model weights and Imputers loaded.')
except FileNotFoundError:
    raise RuntimeError('Models not found. Run training.ipynb first.')

## 2. Data Ingestion & Pre-processing

In [ ]:
# Load incoming data (in production, replace with live data source)
df_infer = pd.read_csv(os.path.join('../test_data', 'dataset_daily_test.csv'))
df_infer['Date'] = pd.to_datetime(df_infer['Date'])

# KNN imputation for missing values (No Leakage: Imputer was fitted tightly on original train data)
df_infer['Time_Idx'] = df_infer['Date'].dt.dayofyear
features_to_impute = ['Time_Idx', 'Demand_MWh', 'Avg_Temp', 'Rainfall']

# Ensure the columns exist so we can transform
for c in features_to_impute:
    if c not in df_infer.columns:
        df_infer[c] = np.nan
df_infer[features_to_impute] = knn_imputer.transform(df_infer[features_to_impute])
df_infer.drop(columns=['Time_Idx'], inplace=True)

# --- Derive additional temporal and autoregressive features ---
# These must match the feature engineering in hybrid_model.py
df_infer['Month']      = df_infer['Date'].dt.month
df_infer['DayOfYear']  = df_infer['Date'].dt.dayofyear
df_infer['WeekOfYear'] = df_infer['Date'].dt.isocalendar().week.astype(int)
df_infer['Trend']      = (df_infer['Date'] - pd.Timestamp('2018-01-01')).dt.days
df_infer['Lag_2']      = df_infer[TARGET_COL].shift(2)
df_infer['Lag_14']     = df_infer[TARGET_COL].shift(14)
df_infer['Rolling_14'] = df_infer[TARGET_COL].rolling(window=14, min_periods=1).mean()
df_infer['Rolling_30'] = df_infer[TARGET_COL].rolling(window=30, min_periods=1).mean()
df_infer['Temp_Lag_1'] = df_infer['Avg_Temp'].shift(1)

# Resolve available features
features = [c for c in FEATURE_CANDIDATES if c in df_infer.columns]
df_clean = df_infer.dropna(subset=features).copy()
print(f'Loaded {len(df_clean)} rows with features: {features}')

## 3. Anomaly Detection & Imputation

In [ ]:
# Isolation Forest anomaly scoring
if_predictions = iso_forest.predict(df_clean[features])

# IQR anomaly detection on target (only when target column is available)
if TARGET_COL in df_clean.columns:
    q1 = df_clean[TARGET_COL].quantile(0.25)
    q3 = df_clean[TARGET_COL].quantile(0.75)
    iqr_val = q3 - q1
    iqr_anomalies = np.where(
        (df_clean[TARGET_COL] < q1 - 1.5 * iqr_val) | (df_clean[TARGET_COL] > q3 + 1.5 * iqr_val),
        -1, 1,
    )
    is_anomaly = (if_predictions == -1) | (iqr_anomalies == -1)

    # Impute anomalous target values with trailing 7-day clean mean
    for idx in np.where(is_anomaly)[0]:
        start = max(0, idx - 7)
        clean_mask = ~is_anomaly[start:idx]
        clean_window = df_clean[TARGET_COL].iloc[start:idx][clean_mask]
        imputed = clean_window.mean() if len(clean_window) > 0 else df_clean[TARGET_COL].mean()
        df_clean.iloc[idx, df_clean.columns.get_loc(TARGET_COL)] = imputed

    print(f'Imputed {is_anomaly.sum()} anomalies via IQR + Isolation Forest.')
else:
    is_anomaly = if_predictions == -1
    print(f'No target column — flagged {is_anomaly.sum()} anomalies via Isolation Forest only.')

df_clean['Anomaly_Flag'] = np.where(is_anomaly, 'ALERT', 'OK')

## 4. Hybrid Forecast

In [ ]:
# Prophet baseline (with temperature regressor)
df_prophet = df_clean[['Date', 'Avg_Temp']].rename(columns={'Date': 'ds'})
prophet_preds = prophet_model.predict(df_prophet)['yhat'].values

# LightGBM residual corrections
lgbm_preds = lgbm_model.predict(df_clean[features])

# Final hybrid combination
df_clean['Forecast_MWh'] = prophet_preds + lgbm_preds

print('--- INFERENCE PREVIEW ---')
display(df_clean[['Date', 'Forecast_MWh', 'Anomaly_Flag']].tail(10))

## 5. Explainability — SHAP Narrative (Top-3 Feature Impacts)

In [ ]:
print('\n--- EXPLAINABILITY (SHAP) ON LATEST INPUT ---')

# Compute SHAP values for all inference inputs
explainer = shap.TreeExplainer(lgbm_model)
shap_vals = explainer.shap_values(df_clean[features])

# Focus on the latest (most recent) data point
latest_shap = shap_vals[-1]
latest_input = df_clean[features].iloc[-1]
latest_date  = df_clean['Date'].iloc[-1]

feature_impacts = list(zip(features, latest_shap, latest_input.values))
feature_impacts.sort(key=lambda x: abs(x[1]), reverse=True)

print(f'\nTanggal inferensi terakhir: {latest_date:%Y-%m-%d}')
print(f'Prediksi akhir: {df_clean["Forecast_MWh"].iloc[-1]:,.0f} MWh')
print(f'\nTop-3 Faktor Penentu Keputusan AI:')
print('=' * 70)

for i, (feat, impact, val) in enumerate(feature_impacts[:3]):
    direction = 'Meningkatkan prediksi (Positif ↑)' if impact > 0 else 'Menurunkan prediksi (Negatif ↓)'
    print(f'{i+1}. {feat} (nilai = {val:.2f})')
    print(f'   ➤ {direction}')
    print(f'   ➤ Besar dampak: {abs(impact):,.2f} MWh')
    print()

print('=' * 70)
print('\nNarasi lengkap:')
parts = []
for i, (feat, impact, val) in enumerate(feature_impacts[:3]):
    arah = 'meningkatkan' if impact > 0 else 'menurunkan'
    parts.append(f'{feat} (nilai={val:.2f}) {arah} prediksi sebesar {abs(impact):,.2f} MWh')
narasi = (f'Pada {latest_date:%Y-%m-%d}, tiga faktor utama yang mempengaruhi '
          f'prediksi demand listrik adalah: {parts[0]}; {parts[1]}; dan {parts[2]}.')
print(narasi)

## 6. Export Results

In [ ]:
output_path = os.path.join(OUTPUT_DIR, 'inference_results.csv')
df_clean.to_csv(output_path, index=False)
print(f'\nInference complete. Output saved to: {output_path}')